# Lastde++ — Training-Free Detection by Mining Token Probability Sequences

Xu et al., **ICLR 2025** · [iclr.cc/virtual/2025/poster/27889](https://iclr.cc/virtual/2025/poster/27889) ·
arXiv: https://arxiv.org/abs/2410.06072 · code: https://github.com/TrustMedia-zju/Lastde_Detector

> **Phạm vi**: chỉ inference + eval theo bộ metric của supervised (AUROC/TPR@FPR/FPR@τ trên
> `val`/`test_ood_model`/`test_fp`). Chưa eval dưới attack. Nhánh **DetectGPT/Fast-DetectGPT**,
> không train.
>
> **Model 3/3 của nhánh** — thay cho `detect_03_glimpse.ipynb`. Glimpse cần `echo=True` +
> top-K logprobs trên `/v1/completions`; OpenRouter có **0/364** model hỗ trợ `echo`, nên không
> chạy được bằng key hiện có. Lastde++ giữ nguyên vị trí narrative (kế thừa trực tiếp criterion
> sampling discrepancy của Fast-DetectGPT) nhưng **chạy hoàn toàn local, không cần API**.

## Cơ chế

Lastde khai thác **động lực học cục bộ** của chuỗi log-probability (TPS) — thứ mà mọi baseline
trước đó bỏ qua vì chỉ dùng thống kê toàn cục (likelihood, rank, entropy).

```
1. TPS:          L(t) = (log p(t_1|t_<1), ..., log p(t_n|t_<n))                        (Eq. 1)
2. Multiscale:   L^(τ)(j) = (1/τ) Σ_{i=j}^{j+τ-1} log p(t_i|t_<i),  τ = 1..τ'          (Eq. 2)
3. Sliding:      cắt L^(τ) bằng cửa sổ s, step 1 -> các s-probability segment           (Eq. 3-4)
4. Similarity:   d_k = CosineSimilarity(l_k(s), l_k+1(s))  giữa 2 segment liền kề       (Eq. 5)
5. State:        chia [-1,1] thành ε khoảng -> histogram -> chuỗi xác suất P^(τ)        (Eq. 6)
6. DE:           DE(s,ε,τ) = -(1/ln ε) Σ_i P_i^(τ) ln P_i^(τ)                           (Eq. 8)
7. MDE:          MDE = (DE(s,ε,1), ..., DE(s,ε,τ'))                                     (Eq. 7)
8. Lastde:       Lastde(t,θ) = [ (1/n) Σ log p(t_i|t_<i) ] / Agg(MDE)                   (Eq. 9)
                              ^^^ thống kê TOÀN CỤC       ^^^ thống kê CỤC BỘ
9. Lastde++:     D(t,θ,θ̃) = (Lastde(t,θ) − μ̃) / σ̃,  μ̃/σ̃ lấy từ 100 mẫu ~ M_θ̃      (Eq. 10-11)
```

Trực giác: cosine similarity giữa hai đoạn log-prob liền kề đo **độ dao động cục bộ** của TPS.
Văn bản LLM có TPS trơn hơn (ít dao động, DE thấp) so với văn bản người. Chia likelihood cho
Agg-MDE khuếch đại đúng sự khác biệt đó, rồi bọc thêm sampling discrepancy của Fast-DetectGPT
để chuẩn hoá theo phân phối.

Vì `θ = θ̃` (cùng gpt-j-6b) nên **sampling và scoring gộp làm một** — chỉ cần **1 forward pass**
mỗi văn bản, rẻ hơn `detect_02_fast_detectgpt.ipynb` (2 forward pass, 2 model).

## Deviation so với paper / code gốc

| # | Paper / code gốc | Ta dùng | Loại | Hệ quả |
|---|---|---|---|---|
| 1 | `--embed_size 4 --epsilon 8 --tau_prime 15` (`shell_scripts/detection_black_box.sh` **và** `detection_white_box.sh`) | **Khớp**: `EMBED_SIZE=4`, `EPSILON_MULT=8`, `TAU_PRIME=15` | — | — |
| 1a | ⚠️ README của tác giả ghi `τ'=10` cho Lastde++ | Dùng **`τ'=15`** | sửa lỗi nguồn | Paper (§3.2, tr. 6), `lastde_doubleplus.py` (`default=15`) và **cả hai** shell script đều là 15. README sai. |
| 2 | `Agg = Std`, hardcode `torch.std(mde, dim=0)` trong `fastMDE.get_tau_multiscale_DE` (unbiased, ddof=1) | **Khớp** `AGG="std"`, giữ ddof=1 | — | `np.std` mặc định ddof=0 -> **không** dùng numpy ở chỗ này |
| 2a | README gợi ý *"we recommend you set Agg to ExpStd"* (paper Appendix D) | Có `AGG="expstd"` nhưng **không bật mặc định** | phạm vi | Giữ đúng default của paper; đổi 1 dòng nếu muốn chạy ablation Appendix D |
| 3 | `--n_samples 100` (1/100 của Fast-DetectGPT) | **Khớp** `N_SAMPLES=100` | — | — |
| 4 | Black-box: `proxy_model="gptj_6b"` cho **cả** `reference` và `scoring` | **Khớp**: `EleutherAI/gpt-j-6b` cho cả hai | — | `θ = θ̃` -> gộp 1 forward pass, đúng ghi chú §3.2 của paper |
| 4a | — | Có thể đổi `SCORING_MODEL="EleutherAI/gpt-neo-2.7B"` | tuỳ chọn | Khi đó khớp **đúng cặp backbone** của `detect_02_fast_detectgpt.ipynb` -> so sánh có kiểm soát Fast-DetectGPT vs Lastde++ (xem cell cấu hình) |
| 5 | Không cắt độ dài | **`MAX_LEN = 512`** token | hạ tầng | Đồng bộ ngân sách đầu vào với `supervised_models/` + Fast-DetectGPT + Glimpse |
| 6 | Statistics tính trên dtype của logits (fp16 với gpt-j) | **fp16 weights, fp32 statistics** | hạ tầng | Đồng bộ `detect_02_fast_detectgpt.ipynb` (`RUN_SIGNATURE` cùng ghi `fp16-weights-fp32-statistics`) |
| 7 | `--seed 0` | **`SEED = 42`** | hạ tầng | Đồng bộ seed toàn nhánh; seed chỉ ảnh hưởng 100 mẫu Categorical |
| 8 | `torch.vmap(histcounts)` + `torch.histc` | Histogram vectorised bằng `scatter_add_` | tăng cường | Hai lý do. (a) `torch.histc` **không có batching rule** trong functorch -> `vmap` rơi về vòng lặp tuần tự (warning bị `filterwarnings("ignore")` che). (b) `vmap` trả tensor **non-contiguous** khiến `torch.nansum(dim=0)` chạy thứ tự reduction khác -> lệch ~1 ulp float32 (1.2e-07) **so với chính `torch.histc` tuần tự**. Cell VERIFY chứng minh port của ta `max\|Δ\| = 0.0` so với `torch.histc` tuần tự, và báo cáo riêng độ lệch của đường `vmap` |
| 9 | 1 process, 1 GPU | Sharding 2 GPU + chunk checkpoint atomic | tăng cường | Kaggle session ngắt không mất việc đã chấm |
| 10 | Chỉ báo AUROC / PR-AUC | Thêm `TPR@FPR{0.1,1,5}%`, `τ@FPR=1%` trên val, `FPR@τ` | mở rộng | Đồng bộ bảng số với `supervised_models/` |

**Không** áp cột attack ở notebook này. **Không** tự lật dấu score theo nhãn val
(`statistical_branch_plan.md` §4) — hướng `score cao = AI` được **đăng ký trước** theo định nghĩa
paper (`get_roc_metrics(real_preds, sample_preds)` với `real`=human nhãn 0, `samples`=AI nhãn 1),
chỉ cảnh báo nếu `AUROC(val) < 0.5`.

## ⚖️ Công bằng khi đặt cạnh supervised + AL

### Phần protocol ĐÃ đồng nhất (đã đối chiếu từng dòng với `supervised_models/train_01_roberta.ipynb`)

| Trục | supervised / AL | Lastde++ | Khớp? |
|---|---|---|---|
| Split & cách melt | `melt_pairs` / `melt_fp` | y hệt | ✅ |
| `MAX_LEN` | 512 | 512 | ✅ |
| Công thức τ | `np.quantile(neg, 1-fpr, method="higher")` trên `val` | **giống từng ký tự** | ✅ |
| `TARGET_FPR` | 0.01 | 0.01 | ✅ |
| Bộ metric | AUROC, AUPRC, TPR@FPR{0.1,1,5}%, TPR@τ, FPR@τ | y hệt | ✅ |
| `sample_weight` khi **eval** | **không dùng** (chỉ vào `compute_loss` lúc train) | không dùng | ✅ |
| Schema `metrics.json` | `run/model/tau/target_fpr/world_size/seed/metrics[]` | y hệt | ✅ |
| Layout artifacts | `artifacts_{RUN_NAME}_seed{SEED}` | y hệt | ✅ |
| Env seed | `RUN_SEED`, mặc định 42 | y hệt | ✅ |
| Multi-seed `mean ± std` | có | **có** (xem cell cuối) | ✅ |

### 3 điểm bất đối xứng PHẢI ghi rõ — không sửa được bằng code

**1. `val` bị supervised dùng để CHỌN CHECKPOINT ⇒ không xếp hạng trên `val`.**
Cả 4 notebook supervised đều `load_best_model_at_end=True`, `metric_for_best_model="auroc"` trên
`val`, cộng `EarlyStoppingCallback`. Supervised **fit lên `val`**; Lastde++ chỉ lấy `τ` từ `val`.
⇒ `AUROC(val)` của supervised **lạc quan có hệ thống**. Headline so sánh phải đặt trên
**`test_ood_model`** và **`test_fp`**.

**2. `test_ood_model` là OOD với supervised nhưng KHÔNG OOD với Lastde++.**
`train`/`val` gồm `openai/mistral/qwen/anthropic/meta`; `test_ood_model` là **`google`/`deepseek`**.
Với supervised đó là OOD thật. Với Lastde++ training-free thì **không tồn tại in-domain theo
generator**. ⇒ `Δ AUROC(val→OOD)` là đại lượng **chỉ có nghĩa với supervised**. Lastde++ có `Δ ≈ 0`
không phải vì robust hơn, mà vì định nghĩa OOD không áp dụng cho nó.

**3. Lastde++ ở đây là BLACK-BOX — nhưng đúng setting mà paper thiết kế.**
Proxy là `gpt-j-6b`, không có dòng `ai_text` nào do GPT-J sinh. Khác DetectGPT (nơi `gpt2-xl` là
lựa chọn ứng biến), đây **chính là** cấu hình `detection_black_box.sh` của tác giả
(`proxy_model="gptj_6b"`), và paper báo cáo black-box **đối đầu cả closed-source**
(`gpt4turbo`/`gpt4o`/`claude3haiku`) bằng đúng proxy này. ⇒ Caveat vẫn phải nêu, nhưng **nhẹ hơn**
DetectGPT: số đọc là *"black-box, đúng protocol tác giả"*, không phải *"black-box ứng biến"*.

*Quy mô*: `roberta-large` 355M **được train** vs Lastde++ `gpt-j-6b` **6B frozen, 1 model,
1 forward pass**. So với DetectGPT (4.3B = `gpt2-xl` 1.5B + `t5-3b` 2.8B, 2 model) thì Lastde++
nặng hơn về params nhưng **rẻ hơn nhiều về compute**. Nếu bảng có cột params/compute thì ghi rõ cả hai.

*Lưu ý tokenizer*: `gpt-j` dùng BPE GPT-2, còn `deberta-v3`/`xlm-r` dùng sentencepiece — "512 token"
của sentencepiece là **nhiều text hơn** ~10-15%. Đây là bất đồng nhất có sẵn **giữa các supervised
model với nhau** (chuẩn mực literature: mỗi model dùng `max_length` của chính nó), Lastde++ không bị
đối xử khác. Phát biểu đúng là *"cùng quy tắc 512 token"*, không phải *"cùng lượng text"*.

### Về số seed — đây là chỗ Lastde++ **khớp được** với supervised, khác DetectGPT

Supervised chạy nhiều `RUN_SEED` rồi báo `mean ± std`. DetectGPT không làm nổi vì 1 seed đầy đủ
≈ 18h (3 seed = 54h > quota 30h/tuần) nên phải ghi `ood`/`fp` là single-seed. Lastde++ full chỉ
**~1-2.5h** ⇒ **3 seed full ≈ 3-7.5h, vừa quota**. Seed ảnh hưởng 100 mẫu Categorical, nên
`mean ± std` ở đây là đại lượng có nghĩa thật. Chạy `RUN_SEED=1` / `RUN_SEED=2` ở cell FULL rồi
dùng cell tổng hợp cuối notebook.


In [ ]:
!pip install -q -U "transformers>=4.45" accelerate sentencepiece

In [ ]:
%%writefile lastde_core.py
"""Phần toán của Lastde/Lastde++ — port từ py_scripts/baselines/scoring_methods/fastMDE.py
   (https://github.com/TrustMedia-zju/Lastde_Detector).

Khác biệt duy nhất so với code gốc: thay `torch.vmap(histcounts)` bằng histogram vectorised
(`scatter_add_`), vì `torch.histc` không có batching rule nên `vmap` rơi về vòng lặp tuần tự.
Ngữ nghĩa được giữ y nguyên và cell VERIFY của notebook chứng minh max|Δ| = 0.
"""
import math

import torch


def histogram_probs(x, eps_bins):
    """Tương đương `torch.histc(x[..., s], bins=eps_bins, min=-1, max=1)` cho từng lát của
    chiều cuối, rồi chuẩn hoá về tổng 1.

    x: [..., S] (float)  ->  [eps_bins, S]

    Bám đúng 2 quy ước của torch.histc:
      * phần tử ngoài [min, max] bị **bỏ** (không dồn vào bin biên),
      * bin = min(floor((x - min) / (max - min) * bins), bins - 1)  -> x == max vào bin cuối.
    """
    S = x.shape[-1]
    flat = x.reshape(-1, S).float()
    keep = (flat >= -1.0) & (flat <= 1.0)
    idx = torch.floor((flat + 1.0) * (eps_bins / 2.0)).long().clamp_(0, eps_bins - 1)
    col = torch.arange(S, device=x.device).expand_as(idx)
    lin = (idx * S + col)[keep]
    hist = torch.zeros(eps_bins * S, device=x.device, dtype=torch.float32)
    hist.scatter_add_(0, lin, torch.ones_like(lin, dtype=torch.float32))
    hist = hist.view(eps_bins, S)
    return hist / hist.sum(dim=0, keepdim=True)


def de_from_probs(probs, eps_bins):
    """DE(s, ε, τ) = -(1/ln ε) Σ_i P_i ln P_i,  chỉ tính trên P_i > 0.   (Eq. 8)

    Giữ NGUYÊN VĂN biểu thức của code gốc, kể cả các chi tiết ảnh hưởng bit cuối:
      * `torch.nansum` để bỏ bin rỗng (0 * log 0 = nan), không dùng `where`;
      * `-1 / torch.log(torch.tensor(eps_bins))` rồi NHÂN — không dùng `math.log` (float64)
        rồi chia, vì lệch ~3e-08 và cell VERIFY sẽ báo không bit-identical.
    """
    return -1 / torch.log(torch.tensor(eps_bins)) * torch.nansum(
        probs * torch.log(probs), dim=0)


def calculate_DE(seq, embed_size, eps_bins):
    """seq: [1, T, S] -> DE cho từng sample: [S].   (Eq. 3-6, 8)"""
    orbits = seq.unfold(1, embed_size, 1)                       # [1, T-s+1, S, s]
    cos = torch.nn.functional.cosine_similarity(
        orbits[:, :-1], orbits[:, 1:], dim=-1)                  # [1, T-s, S]
    return de_from_probs(histogram_probs(cos, eps_bins), eps_bins)


def agg_mde(loglik, embed_size, eps_bins, tau_prime, agg="std"):
    """Agg(MDE) — mẫu số của Eq. 9.  loglik: [1, n, S] -> [S].

    `torch.std` mặc định unbiased=True (ddof=1), khớp code gốc. KHÔNG thay bằng np.std (ddof=0).
    """
    des = []
    for tau in range(1, tau_prime + 1):
        scaled = loglik.unfold(1, tau, 1).mean(dim=3)            # Eq. 2: [1, n-tau+1, S]
        des.append(calculate_DE(scaled, embed_size, eps_bins))
    mde = torch.stack(des, dim=0)                                # [tau_prime, S]
    out = torch.std(mde, dim=0)                                  # ddof=1
    return torch.exp(out) if agg == "expstd" else out


def lastde(loglik, embed_size, eps_mult, tau_prime, agg="std"):
    """Lastde(t, θ) = mean(log p) / Agg(MDE).   (Eq. 9)   loglik: [1, n, S] -> [1, S].

    ε = eps_mult * n với n = số token, đúng `int(args.epsilon * log_likelihood.shape[1])`.
    """
    n = loglik.shape[1]
    eps_bins = int(eps_mult * n)
    return loglik.mean(dim=1) / agg_mde(loglik, embed_size, eps_bins, tau_prime, agg)


def min_tokens_required(embed_size, tau_prime):
    """Cần ít nhất 2 segment liền kề ở scale lớn nhất: n - τ' - s + 2 >= 2."""
    return tau_prime + embed_size


## VERIFY — port có khớp code gốc không?

Cell dưới dựng lại **nguyên văn** `fastMDE.py` của tác giả (kể cả `torch.vmap` + `torch.histc`)
rồi so với `lastde_core.py`. Chạy trên CPU, vài giây, **không cần GPU**.

Đây là cùng quy trình đã dùng cho Glimpse (`statistical_branch_plan.md` §9: *"port nguyên văn,
đã verify bit-identical"*). Nếu `max|Δ|` không phải 0 thì **dừng lại**, đừng chạy FULL.


In [ ]:
# ── VERIFY: lastde_core.py vs fastMDE.py nguyên văn của tác giả ───────────────
import warnings

import torch

import lastde_core

warnings.filterwarnings("ignore")
torch.manual_seed(0)


# ── Nguyên văn từ py_scripts/baselines/scoring_methods/fastMDE.py ─────────────
def _ref_histcounts(data, epsilon, min_=-1, max_=1):
    data = data.float()
    hist = torch.histc(data, bins=epsilon, min=min_, max=max_)
    return hist, hist / torch.sum(hist)


def _ref_DE(statistical_probabilities_sequence, epsilon):
    return -1 / torch.log(torch.tensor(epsilon)) * torch.nansum(
        statistical_probabilities_sequence * torch.log(statistical_probabilities_sequence), dim=0)


def _ref_calculate_DE(ori_data, embed_size, epsilon, use_vmap=True):
    orbits = ori_data.unfold(1, embed_size, 1)
    cos = torch.nn.functional.cosine_similarity(orbits[:, :-1], orbits[:, 1:], dim=-1)
    if use_vmap:
        batched_1 = torch.vmap(_ref_histcounts, in_dims=-1, out_dims=1)
        _, probs = batched_1(cos, epsilon=epsilon)
    else:  # đường tham chiếu độc lập: histc tuần tự từng sample, không qua vmap
        cols = [_ref_histcounts(cos[..., s], epsilon)[1] for s in range(cos.shape[-1])]
        probs = torch.stack(cols, dim=1)
    return _ref_DE(probs, epsilon)


def _ref_get_tau_scale_DE(ori_data, embed_size, epsilon, tau, use_vmap=True):
    windows = ori_data.unfold(1, tau, 1)
    return _ref_calculate_DE(torch.mean(windows, dim=3), embed_size, epsilon, use_vmap)


def _ref_get_tau_multiscale_DE(ori_data, embed_size, epsilon, tau_prime, use_vmap=True):
    mde = torch.stack([_ref_get_tau_scale_DE(ori_data, embed_size, epsilon, t, use_vmap)
                       for t in range(1, tau_prime + 1)], dim=0)
    return torch.std(mde, dim=0)


# ── So sánh trên nhiều hình dạng, gồm cả cạnh biên ────────────────────────────
EMBED_SIZE, EPSILON_MULT, TAU_PRIME = 4, 8, 15
_vmap_ok = True
try:
    torch.vmap(_ref_histcounts, in_dims=-1, out_dims=1)(torch.randn(1, 40, 3), epsilon=32)
except Exception as _ex:  # noqa: BLE001
    _vmap_ok = False
    print(f"[verify] torch.vmap(histc) khong dung duoc ({type(_ex).__name__}) "
          f"-> doi sang tham chieu histc tuan tu.")

_cases = [(1, 512, 1), (1, 512, 100), (1, 130, 100), (1, 64, 8),
          (1, TAU_PRIME + EMBED_SIZE, 4)]  # case cuoi = do dai toi thieu
_worst_seq, _worst_vmap = 0.0, 0.0
print("cot 'seq'  = so voi torch.histc tuan tu  -> ground truth, phai = 0")
print("cot 'vmap' = so voi torch.vmap(histc)    -> chi bao cao, xem ghi chu duoi\n")
for _b, _n, _s in _cases:
    x = torch.randn(_b, _n, _s, dtype=torch.float32) * 3.0 - 6.0   # thang do giong log-prob
    _eps_bins = int(EPSILON_MULT * _n)
    got = lastde_core.agg_mde(x, EMBED_SIZE, _eps_bins, TAU_PRIME, "std")
    exp_seq = _ref_get_tau_multiscale_DE(x, EMBED_SIZE, _eps_bins, TAU_PRIME, use_vmap=False)
    d_seq = (got - exp_seq).abs().max().item()
    _worst_seq = max(_worst_seq, d_seq)
    if _vmap_ok:
        exp_vmap = _ref_get_tau_multiscale_DE(x, EMBED_SIZE, _eps_bins, TAU_PRIME, use_vmap=True)
        d_vmap = (got - exp_vmap).abs().max().item()
        _worst_vmap = max(_worst_vmap, d_vmap)
    else:
        d_vmap = float("nan")
    print(f"[verify] shape={(_b, _n, _s)} eps_bins={_eps_bins:>5}  "
          f"seq: {d_seq:.3e}   vmap: {d_vmap:.3e}")

# Kiem tra rieng histogram: phai trung khop torch.histc ke ca khi co gia tri ngoai [-1,1]
_edge = torch.tensor([-1.0, 1.0, -1.0000001, 1.0000001, 0.0, 0.5]).reshape(1, -1, 1)
_hp = lastde_core.histogram_probs(_edge, 10)
_hr = _ref_histcounts(_edge[..., 0], 10)[1][:, None]
_d_edge = (_hp - _hr).abs().max().item()
print(f"\n[verify] histogram bien (gom ca gia tri ngoai [-1,1]): max|delta| = {_d_edge:.3e}")

assert _worst_seq == 0.0 and _d_edge == 0.0, (
    f"Port KHONG bit-identical voi torch.histc tuan tu (max|delta| = "
    f"{max(_worst_seq, _d_edge):.3e}). Dung lai, dung chay FULL.")
print(f"\n[verify] OK — bit-identical (max|delta| = 0.0) voi fastMDE.py goc "
      f"tren {len(_cases)} truong hop + case bien.")
print("[verify] torch.std ddof=1 (unbiased) da duoc giu; np.std (ddof=0) KHONG duoc dung.")
if _vmap_ok and _worst_vmap > 0:
    print(f"\n[verify] GHI CHU: duong torch.vmap(histc) lech {_worst_vmap:.3e} so voi ca hai.")
    print("[verify] Da truy nguyen: chuoi xac suat P giong nhau BIT-FOR-BIT o ca 3 duong;")
    print("[verify] vmap tra tensor non-contiguous nen torch.nansum(dim=0) chay thu tu reduction")
    print("[verify] khac -> lech ~1 ulp cua float32. Day la nhieu cua vmap, KHONG phai cua port.")
    print("[verify] Do la ly do thu hai (ngoai toc do) de thay vmap — xem deviation #8.")


In [ ]:
# ── VERIFY 2: toàn bộ chuỗi tính điểm (logits -> discrepancy) vs bản gốc ─────
# Cell trên chỉ kiểm phần MDE. Cell này kiểm luôn phần ghép: sampling Categorical,
# gather log-likelihood, mean/std 100 mẫu — tức đúng những gì lastde_worker.score() làm.
# Chạy CPU, vài giây, không cần tải model.
import statistics

import torch

import lastde_core

EMBED_SIZE, EPSILON_MULT, TAU_PRIME, N_SAMPLES = 4, 8, 15, 100


# ── nguyên văn get_samples/get_likelihood/get_lastde/get_sampling_discrepancy ──
def _ref_mde(ori_data, embed_size, epsilon, tau_prime):
    def _hist(data, eps):
        h = torch.histc(data.float(), bins=eps, min=-1, max=1)
        return h / torch.sum(h)

    def _de(ori, eps):
        orb = ori.unfold(1, embed_size, 1)
        cos = torch.nn.functional.cosine_similarity(orb[:, :-1], orb[:, 1:], dim=-1)
        P = torch.stack([_hist(cos[..., s], eps) for s in range(cos.shape[-1])], dim=1)
        return -1 / torch.log(torch.tensor(eps)) * torch.nansum(P * torch.log(P), dim=0)

    mde = torch.stack([_de(ori_data.unfold(1, t, 1).mean(dim=3), epsilon)
                       for t in range(1, tau_prime + 1)], dim=0)
    return torch.std(mde, dim=0)


def _ref_lastde(ll):
    eps = int(EPSILON_MULT * ll.shape[1])
    return ll.mean(dim=1) / _ref_mde(ll, EMBED_SIZE, eps, TAU_PRIME)


def ref_discrepancy(logits_ref, logits_score, labels):
    lprobs = torch.log_softmax(logits_ref, dim=-1)
    samples = torch.distributions.categorical.Categorical(
        logits=lprobs).sample([N_SAMPLES]).permute([1, 2, 0])
    lp_s = torch.log_softmax(logits_score, dim=-1)
    ll_x = lp_s.gather(dim=-1, index=labels.unsqueeze(-1))
    ll_t = lp_s.gather(dim=-1, index=samples)
    sampled = _ref_lastde(ll_t)
    return ((_ref_lastde(ll_x) - sampled.mean()) / sampled.std()).cpu().item()


# ── chuỗi mà lastde_worker.score() dùng ───────────────────────────────────────
def worker_discrepancy(logits_ref, logits_score, labels):
    lprobs_ref = torch.log_softmax(logits_ref.float(), dim=-1)
    samples = torch.distributions.categorical.Categorical(
        logits=lprobs_ref).sample([N_SAMPLES]).permute([1, 2, 0])
    lprobs_score = torch.log_softmax(logits_score.float(), dim=-1)
    ll_x = lprobs_score.gather(dim=-1, index=labels.unsqueeze(-1))
    ll_t = lprobs_score.gather(dim=-1, index=samples)
    lastde_x = lastde_core.lastde(ll_x, EMBED_SIZE, EPSILON_MULT, TAU_PRIME, "std")
    lastde_t = lastde_core.lastde(ll_t, EMBED_SIZE, EPSILON_MULT, TAU_PRIME, "std")
    return float(((lastde_x - lastde_t.mean()) / lastde_t.std()).reshape(-1)[0].item())


VOCAB = 512
_worst = 0.0
for _trial, _n in enumerate([300, 128, 40, EMBED_SIZE + TAU_PRIME]):
    _logits = torch.randn(1, _n, VOCAB, dtype=torch.float32) * 2.0
    _labels = torch.randint(0, VOCAB, (1, _n))
    torch.manual_seed(1234 + _trial)
    _a = ref_discrepancy(_logits, _logits, _labels)
    torch.manual_seed(1234 + _trial)
    _b = worker_discrepancy(_logits, _logits, _labels)
    _worst = max(_worst, abs(_a - _b))
    print(f"[pipeline] n_tok={_n:>4}  goc={_a: .10f}  worker={_b: .10f}  |delta|={abs(_a - _b):.3e}")

assert _worst == 0.0, f"Chuoi tinh diem KHONG khop ban goc (max|delta| = {_worst:.3e})"
print(f"\n[pipeline] OK — discrepancy khop ban goc bit-for-bit tren 4 do dai (max|delta| = 0.0)")

# Sanity check huong diem da dang ky truoc: TPS tron (giong LLM) phai cho score CAO hon.
torch.manual_seed(7)
_hi, _lo = [], []
for _ in range(24):
    _peaked = torch.randn(1, 200, VOCAB) * 6.0          # phan phoi nhon ~ LLM
    _flat = torch.randn(1, 200, VOCAB) * 0.5            # phan phoi bet  ~ human
    _hi.append(worker_discrepancy(_peaked, _peaked, _peaked.argmax(-1)))
    _lo.append(worker_discrepancy(_flat, _flat, torch.randint(0, VOCAB, (1, 200))))
print(f"[direction] score 'AI-like'    mean = {statistics.mean(_hi): .4f}")
print(f"[direction] score 'human-like' mean = {statistics.mean(_lo): .4f}")
assert statistics.mean(_hi) > statistics.mean(_lo), "Huong diem sai so voi dang ky 'cao = AI'."
print("[direction] OK — 'cao = AI' dung nhu huong da dang ky truoc.")


In [ ]:
%%writefile lastde_worker.py
import gc
import hashlib
import math
import os
import random
import time
import warnings

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import lastde_core

warnings.filterwarnings("ignore")

# ── Bản dataset dùng để train/eval ────────────────────────────────────
#   "deconf2"    = 02c_deconfound_v2.ipynb  <- ĐANG DÙNG
#                  canonicalize + truncate_pair_sentences + lọc notation_gap 85%.
#                  shortcut_audit.py đo đủ 22 đặc trưng: tầng process 0.5055 (cổng
#                  cứng, PASS) · tổng 0.8909. Train 7.025 cặp (85% của 8.265).
#   "deconf"     = 02b_deconfound.ipynb. ĐỪNG dùng cho nhánh này. Hai lý do:
#                  (a) con số "surface-GBM 0.6867, G6 PASS" mà 02b tự báo chỉ đo 6
#                      đặc trưng, thiếu avg_word_len (0.7795) lẫn type_token_ratio
#                      (0.7727). Đo đủ 22 thì là 0.9114 -> WARN, không PASS.
#                  (b) truncate_pair cắt thô đúng từ thứ min(len) nên chỉ 52,4% văn
#                      bản còn kết thúc đúng dấu câu (trước khi cắt: 99,0%). Detector
#                      ở file này chấm điểm bằng log-probability của từng token, nên
#                      token bị chặt giữa câu là perplexity GIẢ — và nó BẤT ĐỐI XỨNG,
#                      vì câu AI dài hơn (22,4 vs 20,4 từ) nên bên AI mất nhiều từ
#                      hơn một cách có hệ thống.
#   "normalized" = bản GỐC, CÒN shortcut: chỉ ĐẾM SỐ TỪ đã đoán đúng human/AI ở
#                  AUROC 0.7193 (surface-GBM 0.8080). G6 FAIL.
#
# PROVENANCE của số cũ (REPORT_2026-08-01.md §5) — đã truy được, ĐỪNG suy diễn lại:
# log còn lưu trong output notebook ghi sha256 val=c9d3957ae20b ood=2e9a47ca22e0
# fp=584cb43da181, khớp ĐÚNG dataset_{val,test_ood_model,test_fp}_normalized.parquet.
# Tức mọi số static/detectgpt trong báo cáo đó chạy trên "normalized" — chưa
# canonicalize, chưa cắt cặp — KHÔNG phải "deconf" như dòng tag cũ ghi (dòng đó nằm
# trên %%writefile nên chưa bao giờ tới được worker).
# Hệ quả: AUROC(val) < 0.5 của Fast-DetectGPT (0.4760) và Binoculars (0.4650) KHÔNG
# do tiền xử lý. Nguyên nhân phải tìm ở chỗ khác — mismatch thời đại backbone (gpt2-xl
# 2019 / gpt-j 2021 / falcon 2023 vs generator 2026), hoặc bug implementation. PHẢI có
# positive control trên một tập đã biết kết quả công bố trước khi tin bảng so sánh.
#
# PHẢI trùng tag với supervised_models/ và adversarial_models/ — cả 8 notebook đó đã
# chạy trên "deconf2". Lệch tag thì hai bảng số KHÔNG so được với nhau: deconf 8.265
# cặp vs deconf2 7.025 cặp, và nội dung text cũng khác (cắt theo từ vs cắt biên câu).
#
# PHẢI đặt TRONG body của %%writefile, KHÔNG phải ở đầu cell notebook. Đặt ở đầu cell
# thì %%writefile bị đẩy khỏi dòng 0 -> Jupyter hiểu thành line magic và báo "Line
# magic function `%%writefile` not found"; mà kể cả chạy được thì worker vẫn NameError
# vì nó là tiến trình riêng, không thấy biến của notebook.
#
# LƯU Ý: Kaggle dataset phải chứa dataset_{train,val,test_ood_model,test_fp}_deconf2.parquet
DATASET_TAG = "deconf2"
# ── Cấu hình: bám lastde_doubleplus.py + shell_scripts/detection_black_box.sh ──
IMPL_VERSION = "lastde-plusplus-paper-faithful-v1"

# Black-box cua paper: gpt-j-6b lam proxy cho CA reference (M_θ̃) va scoring (M_θ).
# Doi SCORING_MODEL -> "EleutherAI/gpt-neo-2.7B" de khop dung cap backbone cua
# detect_02_fast_detectgpt.ipynb (so sanh co kiem soat); khi do can 2 GPU va WORLD_SIZE=1.
SAMPLING_MODEL = "EleutherAI/gpt-j-6b"
SCORING_MODEL = "EleutherAI/gpt-j-6b"

EMBED_SIZE = 4        # s   — deviation #1
EPSILON_MULT = 8      # ε = EPSILON_MULT * n
TAU_PRIME = 15        # τ'  — 15, KHONG phai 10 (deviation #1a)
N_SAMPLES = 100       # so mau Categorical cho μ̃, σ̃
AGG = "std"           # "std" (paper) | "expstd" (Appendix D)
MAX_LEN = 512         # deviation #5
CHECKPOINT_ROWS = 100
SEED = int(os.environ.get("RUN_SEED", "42"))   # deviation #7; DONG BO ten env voi supervised_models

TIED = SAMPLING_MODEL == SCORING_MODEL
RUN_NAME = "lastde_doubleplus_gptj" if TIED else "lastde_doubleplus_gptj_neo27"
MODEL_DESC = (f"{SAMPLING_MODEL} (theta = theta_tilde)" if TIED
              else f"{SAMPLING_MODEL}/{SCORING_MODEL} (sampling/scoring)")
EVAL_SETTING = "paper_black_box_proxy" if TIED else "fastdetectgpt_paired_backbone"
RUN_SIGNATURE = "|".join([IMPL_VERSION, SAMPLING_MODEL, SCORING_MODEL,
                          f"s{EMBED_SIZE}", f"eps{EPSILON_MULT}n", f"tau{TAU_PRIME}",
                          f"nsamp{N_SAMPLES}", f"agg{AGG}", f"maxlen{MAX_LEN}",
                          f"seed{SEED}", "fp16-weights-fp32-statistics", EVAL_SETTING])

RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
SMOKE_TEST = os.environ.get("SMOKE", "0") == "1"
SMOKE_PER_CLASS = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def log(*a):
    print(f"[lastde++ rank{RANK}]", *a, flush=True)


if not torch.cuda.is_available():
    raise RuntimeError("Can GPU: gpt-j-6b fp16 ~12.1 GiB.")
if TIED:
    # Moi rank giu 1 ban gpt-j tren 1 GPU rieng, chia dong theo RANK (khong can NCCL).
    SCORING_DEVICE = SAMPLING_DEVICE = torch.device(f"cuda:{LOCAL_RANK}")
else:
    if WORLD_SIZE != 1:
        raise RuntimeError("Cau hinh 2 model khac nhau chi chay 1 process (--nproc_per_node=1).")
    if torch.cuda.device_count() < 2:
        raise RuntimeError("Cau hinh 2 model khac nhau can 2 GPU.")
    SAMPLING_DEVICE, SCORING_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")

from transformers import AutoTokenizer, AutoModelForCausalLM

scoring_tok = AutoTokenizer.from_pretrained(SCORING_MODEL)
sampling_tok = scoring_tok if TIED else AutoTokenizer.from_pretrained(SAMPLING_MODEL)
for _tok in {id(scoring_tok): scoring_tok, id(sampling_tok): sampling_tok}.values():
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token

scoring_model = AutoModelForCausalLM.from_pretrained(
    SCORING_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(SCORING_DEVICE).eval()
scoring_model.config.use_cache = False
gc.collect()
torch.cuda.empty_cache()
if TIED:
    sampling_model = scoring_model
else:
    sampling_model = AutoModelForCausalLM.from_pretrained(
        SAMPLING_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(SAMPLING_DEVICE).eval()
    sampling_model.config.use_cache = False
    gc.collect()
    torch.cuda.empty_cache()

MIN_TOKENS = lastde_core.min_tokens_required(EMBED_SIZE, TAU_PRIME)
log("loaded:", MODEL_DESC, "| device:", SCORING_DEVICE)
log("impl:", IMPL_VERSION, "| setting:", EVAL_SETTING)
log(f"s={EMBED_SIZE} eps={EPSILON_MULT}*n tau'={TAU_PRIME} n_samples={N_SAMPLES} "
    f"agg={AGG} max_len={MAX_LEN} | min tokens = {MIN_TOKENS}")
_vocab_crop_logged = False


@torch.inference_mode()
def score(texts):
    """score cao = AI. Sampling Lastde discrepancy, Eq. 10-11.

    Chấm từng văn bản một (giống code gốc): mỗi văn bản có n riêng nên ε = 8n riêng,
    batch chung sẽ phải pad và làm sai n -> sai ε -> sai DE.
    """
    global _vocab_crop_logged
    out = []
    for text in texts:
        enc = scoring_tok(str(text), return_tensors="pt", truncation=True,
                          max_length=MAX_LEN, return_token_type_ids=False)
        n_tok = enc["input_ids"].shape[1] - 1          # sau khi dich nhan 1 buoc
        if n_tok < MIN_TOKENS:
            raise ValueError(
                f"Van ban chi con {n_tok} token sau tokenize, can >= {MIN_TOKENS} "
                f"(tau'={TAU_PRIME} + s={EMBED_SIZE}). Loc bo dong nay hoac giam tau'.")
        score_enc = enc.to(SCORING_DEVICE)
        labels = score_enc["input_ids"][:, 1:]

        score_logits = scoring_model(**score_enc, use_cache=False).logits[:, :-1, :]
        if TIED:
            ref_logits = score_logits
        else:
            samp_enc_cpu = sampling_tok(str(text), return_tensors="pt", truncation=True,
                                        max_length=MAX_LEN, return_token_type_ids=False)
            if not torch.equal(samp_enc_cpu["input_ids"][:, 1:], labels.cpu()):
                raise ValueError("Tokenizer cua 2 model khong khop token IDs.")
            ref_logits = sampling_model(
                **samp_enc_cpu.to(SAMPLING_DEVICE), use_cache=False).logits[:, :-1, :]
            # Code goc cat ve min vocab khi 2 LM co output embedding padding khac nhau.
            v_ref, v_score = ref_logits.size(-1), score_logits.size(-1)
            common = min(v_ref, v_score)
            if int(labels.max().item()) >= common:
                raise ValueError(f"Label token ID vuot common_vocab={common}.")
            if v_ref != v_score and not _vocab_crop_logged:
                log(f"vocab logits mismatch {v_ref} vs {v_score}; crop ca hai ve {common}.")
                _vocab_crop_logged = True
            ref_logits = ref_logits[..., :common].to(SCORING_DEVICE, non_blocking=True)
            score_logits = score_logits[..., :common]

        # deviation #6: statistics tinh o fp32
        lprobs_ref = torch.log_softmax(ref_logits.float(), dim=-1)
        samples = torch.distributions.categorical.Categorical(
            logits=lprobs_ref).sample([N_SAMPLES]).permute([1, 2, 0])   # [1, n, N_SAMPLES]
        del lprobs_ref
        if not TIED:
            del ref_logits

        lprobs_score = torch.log_softmax(score_logits.float(), dim=-1)
        ll_x = lprobs_score.gather(dim=-1, index=labels.unsqueeze(-1))  # [1, n, 1]
        ll_t = lprobs_score.gather(dim=-1, index=samples)               # [1, n, N_SAMPLES]
        del lprobs_score, score_logits, samples

        lastde_x = lastde_core.lastde(ll_x, EMBED_SIZE, EPSILON_MULT, TAU_PRIME, AGG)
        lastde_t = lastde_core.lastde(ll_t, EMBED_SIZE, EPSILON_MULT, TAU_PRIME, AGG)
        # sigma cua 100 mau: torch.std ddof=1, dung nhu sampled_lastde.std() cua code goc
        val = (lastde_x - lastde_t.mean()) / lastde_t.std()
        v = float(val.reshape(-1)[0].item())
        if not math.isfinite(v):
            raise FloatingPointError(
                f"Lastde++ sinh score khong huu han (n_tok={n_tok}). "
                "Agg(MDE) co the bang 0 -> TPS hoan toan phang.")
        out.append(v)
        del ll_x, ll_t, lastde_x, lastde_t
    return np.array(out)


# ── Data + sharding theo RANK + chunk checkpoint (khuôn detect_02_fast_detectgpt) ──
KAGGLE_INPUT = "/kaggle/input/datasets/<kaggle-user>/halfdatadetector"


def find_data_dir():
    target = f"dataset_val_{DATASET_TAG}.parquet"
    if (Path(KAGGLE_INPUT) / target).exists():
        return Path(KAGGLE_INPUT)
    for base in [Path(KAGGLE_INPUT), Path("output"), Path("."), Path("/kaggle/input")]:
        if base.exists():
            hits = list(base.rglob(target))
            if hits:
                return hits[0].parent
    raise FileNotFoundError(f"Khong tim thay {target}. Kiem tra KAGGLE_INPUT = {KAGGLE_INPUT}")


DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}_seed{SEED}"   # DONG BO layout supervised_models
CHUNK_DIR = OUT_DIR / "chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
log("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)


def sha256_file(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def melt_pairs(df):
    h = pd.DataFrame({"text": df["human_text"], "label": 0, "paper_id": df["paper_id"],
                      "ai_degree": 0, "generator_family": "human"})
    a = pd.DataFrame({"text": df["ai_text"], "label": 1, "paper_id": df["paper_id"],
                      "ai_degree": df.get("ai_degree", 3),
                      "generator_family": df.get("generator_family", "unknown")})
    out = pd.concat([h, a], ignore_index=True)
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)


def melt_fp(df):
    has_ai = df["ai_text"].notna() if "ai_text" in df.columns else pd.Series(False, index=df.index)
    text = df["ai_text"].where(has_ai, df["human_text"]) if "ai_text" in df.columns else df["human_text"]
    out = pd.DataFrame({"text": text, "label": 0, "paper_id": df["paper_id"],
                        "ai_degree": df.get("ai_degree", 0).fillna(0).astype(int) if "ai_degree" in df.columns else 0,
                        "generator_family": "human"})
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)


def load(name, fp_mode=False):
    p = DATA_DIR / f"dataset_{name}_{DATASET_TAG}.parquet"
    if not p.exists():
        log(f"thieu {p.name} - bo qua")
        return None
    df = pd.read_parquet(p)
    return melt_fp(df) if fp_mode else melt_pairs(df)


def maybe_smoke(df):
    if df is None or not SMOKE_TEST:
        return df
    n = min(SMOKE_PER_CLASS, int(df["label"].value_counts().min()))
    return (df.groupby("label", group_keys=False)
              .sample(n=n, random_state=SEED)
              .sample(frac=1, random_state=SEED)
              .reset_index(drop=True))


CHUNK_COLS = ["_row_id", "_run_signature", "_data_sha256", "_expected_n", "_chunk_idx", "_rank",
              "paper_id", "label", "ai_degree", "generator_family", "score"]


def validate_chunk(part, path, data_sha, expected_n, chunk_idx, row_ids):
    missing = set(CHUNK_COLS).difference(part.columns)
    ok = (not missing
          and set(part["_run_signature"].unique()) == {RUN_SIGNATURE}
          and set(part["_data_sha256"].unique()) == {data_sha}
          and set(part["_expected_n"].unique()) == {expected_n}
          and set(part["_chunk_idx"].unique()) == {chunk_idx}
          and set(part["_rank"].unique()) == {RANK}
          and part["_row_id"].tolist() == list(row_ids)
          and np.isfinite(part["score"].to_numpy()).all())
    if not ok:
        raise RuntimeError(f"Checkpoint cu/hong: {path}. Doi ten OUT_DIR hoac xoa folder run nay.")


def score_split(split_name, fp_mode=False):
    data_path = DATA_DIR / f"dataset_{split_name}_{DATASET_TAG}.parquet"
    df = maybe_smoke(load(split_name, fp_mode=fp_mode))
    if df is None or len(df) == 0:
        return
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    expected_n = len(df)
    shard = df.iloc[RANK::WORLD_SIZE].reset_index(drop=True)     # chia dong, moi dong doc lap
    if len(shard) == 0:
        return

    if SMOKE_TEST:
        t0 = time.time()
        s = score(shard["text"].tolist())
        log(f"[SMOKE] {split_name}: {len(shard)}/{expected_n} dong, {time.time() - t0:.1f}s, "
            f"finite={np.isfinite(s).all()}, mean={s.mean():.3f}, std={s.std():.3f}; "
            "KHONG luu checkpoint.")
        return

    data_sha = sha256_file(data_path)
    n_chunks = math.ceil(len(shard) / CHECKPOINT_ROWS)
    log(f"{split_name}: {expected_n} dong tong -> shard {len(shard)} dong, {n_chunks} chunks, "
        f"sha256={data_sha[:12]}...")
    split_t0, newly = time.time(), 0
    for chunk_idx in range(n_chunks):
        start = chunk_idx * CHECKPOINT_ROWS
        end = min(start + CHECKPOINT_ROWS, len(shard))
        part = shard.iloc[start:end].copy()
        out_path = CHUNK_DIR / f"{split_name}_rank{RANK}_chunk{chunk_idx:05d}.parquet"
        if out_path.exists():
            done = pd.read_parquet(out_path)
            validate_chunk(done, out_path, data_sha, expected_n, chunk_idx, part["_row_id"])
            log(f"resume {split_name} chunk {chunk_idx + 1}/{n_chunks}: da co {len(done)} dong")
            continue

        t0 = time.time()
        part["score"] = score(part["text"].tolist())
        part["_run_signature"] = RUN_SIGNATURE
        part["_data_sha256"] = data_sha
        part["_expected_n"] = expected_n
        part["_chunk_idx"] = chunk_idx
        part["_rank"] = RANK
        tmp_path = out_path.with_suffix(".tmp.parquet")
        part[CHUNK_COLS].to_parquet(tmp_path, index=False)
        os.replace(tmp_path, out_path)        # atomic: ngat session khong de lai chunk nua voi
        newly += len(part)
        dt = time.time() - t0
        rate = len(part) / max(dt, 1e-9)
        remaining = len(shard) - end
        log(f"saved {split_name} chunk {chunk_idx + 1}/{n_chunks}: {len(part)} dong, "
            f"{rate:.2f} samples/s, ETA moi ~{remaining / max(rate, 1e-9) / 3600:.2f}h")
    log(f"XONG {split_name}: new={newly}, wall={time.time() - split_t0:.1f}s")


for _split, _fp in [("val", False), ("test_ood_model", False), ("test_fp", True)]:
    score_split(_split, fp_mode=_fp)

log(f"peak VRAM {SCORING_DEVICE} = "
    f"{torch.cuda.max_memory_allocated(SCORING_DEVICE.index or 0) / 1024 ** 3:.2f} GiB")
log("XONG SMOKE worker." if SMOKE_TEST else "XONG FULL worker.")

# ═══ CHẾ ĐỘ MIX — chấm tập trộn của 06_mixing để dựng đường dose-response ═════════
# Bật bằng: RUN_MIX=1 <lệnh chạy worker như thường>
#
# Vì sao nằm ở worker này chứ không ở 07_score_mix: 07 load checkpoint bằng
# AutoModelForSequenceClassification. Bốn detector zero-shot KHÔNG có checkpoint
# (falcon/gpt-j đóng băng), nên chúng phải được chấm bằng chính score() ở trên. Và PHẢI là
# score() đó, không viết lại — hai đường trong cùng một figure mà dùng hai hàm chấm khác
# nhau thì figure không nói được gì.
#
# Tập mix KHÔNG đi qua vòng split thường: text đã được 06 assemble sẵn nên không melt, và
# không có label nhị phân — chỉ có mix_ratio_actual liên tục. Vòng split thường tính
# AUROC/τ, vô nghĩa ở đây. Nên mix ghi riêng pred_mix*.parquet và KHÔNG vào metrics.json.
#
# Schema đầu ra = ĐÚNG 15 cột KEEP của 07_score_mix + score -> gộp chung được với
# scores_all.parquet mà không lệch cột.
MIX_NAME = "dataset_test_ood_model_mix.parquet"
MIX_KEEP = ["sample_id", "paper_id", "source", "generator_family", "mix_structure",
            "mix_seed", "mix_k_slots", "n_slots", "mix_ratio_actual", "n_seams",
            "word_count", "align_score", "was_mixed", "ai_degree", "in_spec_band"]
RUN_MIX = os.environ.get("RUN_MIX", "0") == "1"
# 0 = FULL (20.358 dòng). Đặt số n để lấy n dòng MỖI Ô (bin x mix_structure) = 20 ô.
MIX_PER_CELL = int(os.environ.get("MIX_PER_CELL", "0"))


def find_mix_file():
    """mix.parquet ở old_dataset/ khi chạy local, ở /kaggle/input khi chạy Kaggle."""
    for base in [Path(KAGGLE_INPUT), Path("/kaggle/input"), Path("."), Path(".."),
                 Path("old_dataset"), Path("output")]:
        try:
            if base.exists():
                hits = sorted(base.rglob(MIX_NAME))
                if hits:
                    return hits[0]
        except OSError:
            continue
    return None


def load_mix():
    p = find_mix_file()
    if p is None:
        log(f"khong thay {MIX_NAME} -> bo qua che do mix")
        return None, None
    df = pd.read_parquet(p)
    have = [c for c in MIX_KEEP if c in df.columns]
    missing = [c for c in MIX_KEEP if c not in df.columns]
    if missing:
        log(f"[!] mix thieu cot {missing} -> de NaN de concat voi 07 khong lech cot")
    out = df[["text"] + have].copy()
    for c in missing:
        out[c] = np.nan
    out = out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)]
    if MIX_PER_CELL > 0:
        _b = (out["mix_ratio_actual"].fillna(0.0) * 10).astype(int).clip(0, 9)
        out = (out.assign(_bin=_b)
                  .groupby(["_bin", "mix_structure"], group_keys=False)
                  .apply(lambda g: g.sample(n=min(len(g), MIX_PER_CELL), random_state=SEED))
                  .drop(columns="_bin"))
        log(f"mix: subsample {MIX_PER_CELL} dong/o (bin x structure)")
    return out.reset_index(drop=True), p


def run_mix():
    import math as _math
    import time as _time

    _rank = int(globals().get("RANK", 0))
    _ws = int(globals().get("WORLD_SIZE", 1))
    df, src = load_mix()
    if df is None or len(df) == 0:
        return
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    shard = df.iloc[_rank::_ws].reset_index(drop=True) if _ws > 1 else df
    log(f"mix: {len(df)} dong tong -> {len(shard)} dong cua rank {_rank}/{_ws}  <- {src}")
    if SMOKE_TEST:
        _s = score(shard["text"].tolist()[:8])
        log(f"[SMOKE] mix: 8 dong, finite={np.isfinite(_s).all()}, "
            f"min/max={float(np.min(_s)):.4f}/{float(np.max(_s)):.4f}; KHONG luu.")
        return

    ck = OUT_DIR / f"chunks_mix_rank{_rank}"
    ck.mkdir(parents=True, exist_ok=True)
    step = int(globals().get("CHECKPOINT_ROWS", 0) or 200)
    n_ch = _math.ceil(len(shard) / step)
    log(f"mix: {n_ch} chunk x {step} dong, checkpoint tai {ck}")
    t_all = _time.time()
    for k in range(n_ch):
        a, b = k * step, min((k + 1) * step, len(shard))
        p = ck / f"mix_chunk{k:05d}.parquet"
        if p.exists():
            log(f"resume mix chunk {k + 1}/{n_ch}: da co")
            continue
        part = shard.iloc[a:b].copy()
        t0 = _time.time()
        part["score"] = score(part["text"].tolist())
        tmp = p.with_suffix(".tmp.parquet")
        part.drop(columns=["text"]).to_parquet(tmp, index=False)
        os.replace(tmp, p)          # atomic -> het gio session khong de lai chunk viet do dang
        rate = (b - a) / max(_time.time() - t0, 1e-9)
        log(f"mix chunk {k + 1}/{n_ch}: {rate:.2f} mau/s, "
            f"ETA con lai ~{(len(shard) - b) / max(rate, 1e-9) / 3600:.2f}h")

    parts = sorted(ck.glob("mix_chunk*.parquet"))
    if parts:
        allp = pd.concat([pd.read_parquet(x) for x in parts], ignore_index=True)
        dst = OUT_DIR / (f"pred_mix_rank{_rank}.parquet" if _ws > 1 else "pred_mix.parquet")
        allp.to_parquet(dst, index=False)
        log(f"da luu {dst.name}: {len(allp)} dong, wall={_time.time() - t_all:.0f}s")
        log(f"     cot: {list(allp.columns)}")


if RUN_MIX:
    run_mix()

## Chạy trên dual T4 (Kaggle)

Cấu hình mặc định (`SAMPLING_MODEL == SCORING_MODEL == gpt-j-6b`, đúng black-box của paper) chỉ cần
**1 model → 1 forward pass/văn bản**, nên notebook chạy theo lối **sharding**: 2 process độc lập,
mỗi process 1 GPU giữ 1 bản gpt-j (fp16 ~12.1 GiB, vừa T4 16 GiB), mỗi process chấm ~1/2 số dòng.
Không có `dist.init_process_group`/all-reduce nào — chỉ dùng `RANK`/`WORLD_SIZE` để chia việc,
giống `statistical_models/detect_02_dna_gpt.ipynb`.

1. **VERIFY** (cell trên, CPU, vài giây) — bắt buộc chạy trước, phải thấy `bit-identical`.
2. **SMOKE** (`SMOKE=1`, ~1-2 phút) — kiểm 2 GPU không OOM, score hữu hạn.
3. **FULL** — chấm toàn bộ `val`/`test_ood_model`/`test_fp` (8,918 dòng), có chunk checkpoint
   100 dòng nên session ngắt giữa đường thì chạy lại là **resume**, không mất việc.
4. Sau khi cả 2 rank xong → cell **gộp kết quả** → cell **metrics**.

⚠️ Nếu chỉ có **1 GPU**: đổi `--nproc_per_node=2` → `1` (vẫn đúng, chỉ chậm ~2×).

⚠️ Nếu đổi `SCORING_MODEL` sang `gpt-neo-2.7B` (deviation #4a) thì **bắt buộc**
`--nproc_per_node=1` — worker sẽ tự raise nếu bạn quên.


In [ ]:
# ── Checkpoint của lần chạy trước: PHẢI xoá khi đổi DATASET_TAG ───────────────
# Chunk/partial cũ mang _data_sha256 của file parquet CŨ. Notebook nào có
# validate_chunk() thì raise; notebook nào không có thì lẳng lặng gộp score của
# dataset cũ vào metric của dataset mới — đúng loại bug im lặng tệ nhất.
#
# Cell này KHÔNG tự xoá. Nó liệt kê ra để bạn nhìn trước, rồi bạn bật cờ.
import shutil
from pathlib import Path

PURGE_STALE = False   # đặt True MỘT LẦN sau khi đổi tag, chạy cell, rồi trả lại False

_base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
_ckpt, _derived = [], []
for _d in sorted(_base.glob("artifacts_*")):
    _ckpt += [p for p in (_d / "chunks",) if p.is_dir()]
    _ckpt += sorted(_d.glob("partial_*.parquet")) + sorted(_d.glob("chunk_*.parquet"))
    _derived += sorted(_d.glob("pred_*.parquet")) + sorted(_d.glob("metrics*.json"))

if not _ckpt and not _derived:
    print("sạch — không có kết quả của lần chạy trước.")
else:
    print(f"CHECKPOINT (score thô, phải xoá): {len(_ckpt)}")
    for _p in _ckpt:
        print("   ", _p)
    print(f"DẪN XUẤT (metric/pred, bị ghi đè khi chạy lại đủ 3 split): {len(_derived)}")
    for _p in _derived:
        print("   ", _p)
    if PURGE_STALE:
        for _p in _ckpt + _derived:
            shutil.rmtree(_p) if _p.is_dir() else _p.unlink()
        print(f"\nĐÃ XOÁ {len(_ckpt) + len(_derived)} mục.")
    else:
        print("\nPURGE_STALE = False -> chưa xoá gì. Bật True rồi chạy lại cell này.")

In [ ]:
# ── SMOKE: kiểm 2 GPU bắt tay + đọc được parquet, vài phút ──────────────────────
# RUN_MIX=1 ở đây để SMOKE kiểm LUÔN việc tìm thấy dataset_test_ood_model_mix.parquet
# và chấm được 8 dòng của nó. Không có nó thì bạn chỉ phát hiện "quên upload mix" SAU
# khi benchmark đã chạy hàng giờ — worker bỏ qua mix im lặng, không báo lỗi.
# Phải thấy 3 dòng sha256/số dòng của deconf2 VÀ một dòng "[SMOKE] mix: 8 dong".
!RUN_MIX=1 SMOKE=1 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 lastde_worker.py

In [ ]:
# ── CHẠY FULL: 3 split benchmark + tập MIX trong MỘT lần gọi ────────────────────
# RUN_MIX=1 nằm ở đây thay vì một cell riêng vì worker load model ở module level. Tách
# hai lệnh thì Lastde++ load model HAI lần; gộp lại thì load một lần rồi chấm
# splits -> mix liền. Block mix nằm cuối lastde/score_worker.py nên thứ tự tự đúng.
#
# Ước tính: ~53ph + ~53ph = ~1,8h. Dưới hạn 12h/session của Kaggle.
# Checkpoint chunk + atomic replace ở CẢ HAI phần -> session chết thì chạy lại cell này
# là tiếp tục đúng chỗ, không chấm lại gì.
# Muốn rút tập mix: thêm MIX_PER_CELL=150 (150 dòng mỗi ô bin x structure = 3.000 dòng).
!RUN_MIX=1 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 lastde_worker.py

In [ ]:
# ── Gộp chunk của cả 2 rank, kiểm tính toàn vẹn trước khi tính metric ────────
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

import os

RUN_NAME = "lastde_doubleplus_gptj"      # doi thanh ..._gptj_neo27 neu chay cau hinh #4a
TARGET_FPR = 0.01
SEED = int(os.environ.get("RUN_SEED", "42"))
WORLD_SIZE = 2                            # so rank da dung o cell FULL

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}_seed{SEED}"
CHUNK_DIR = OUT_DIR / "chunks"


def merge_split(split_name):
    parts = sorted(CHUNK_DIR.glob(f"{split_name}_rank*_chunk*.parquet"))
    if not parts:
        return None
    d = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

    sigs = set(d["_run_signature"].unique())
    if len(sigs) != 1:
        raise RuntimeError(f"{split_name}: tron nhieu RUN_SIGNATURE {sigs} - xoa chunks va chay lai.")
    shas = set(d["_data_sha256"].unique())
    if len(shas) != 1:
        raise RuntimeError(f"{split_name}: tron nhieu ban dataset {shas} - xoa chunks va chay lai.")

    expected_n = int(d["_expected_n"].iloc[0])
    row_ids = d["_row_id"].to_numpy()
    if len(np.unique(row_ids)) != len(row_ids):
        raise RuntimeError(f"{split_name}: co _row_id trung - 2 rank cham cung dong.")
    missing = sorted(set(range(expected_n)) - set(row_ids.tolist()))
    if missing:
        raise RuntimeError(f"{split_name}: thieu {len(missing)} dong (vi du {missing[:5]}) "
                           f"- chay lai cell FULL de resume.")
    if not np.isfinite(d["score"].to_numpy()).all():
        raise FloatingPointError(f"{split_name}: score co NaN/Inf.")
    return d.sort_values("_row_id").reset_index(drop=True)


val_df = merge_split("val")
ood_df = merge_split("test_ood_model")
fp_df = merge_split("test_fp")
for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if d is None:
        print(f"{nm:<16} CHUA CO chunk - chay cell FULL truoc.")
    else:
        n_rank = d["_rank"].nunique()
        print(f"{nm:<16} {len(d):>7,} mau (gop tu {n_rank} rank)  "
              f"label={d['label'].value_counts().to_dict()}")
if val_df is not None:
    print("\nRUN_SIGNATURE:", val_df["_run_signature"].iloc[0])


In [ ]:
# ── Metrics: CÙNG công thức/schema với supervised_models (tau@FPR=1% trên val) ──
def tpr_at_fpr(y, s, fpr):
    y, s = np.asarray(y), np.asarray(s)
    neg = s[y == 0]
    if len(neg) == 0 or (y == 1).sum() == 0:
        return np.nan, np.nan
    thr = np.quantile(neg, 1 - fpr, method="higher")
    return float((s[y == 1] >= thr).mean()), float(thr)


def auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan


def report_split(name, d):
    if d is None:
        return None
    y, s = d["label"].values, d["score"].values
    row = {"split": name, "n": len(d), "n_ai": int((y == 1).sum()),
           "auroc": auroc(y, s),
           "auprc": float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else np.nan}
    for f in [0.001, 0.01, 0.05]:
        row[f"tpr@fpr{f}"] = tpr_at_fpr(y, s, f)[0]
    row["tpr@tau"] = float((s[y == 1] >= TAU).mean()) if (y == 1).any() else np.nan
    row["fpr@tau"] = float((s[y == 0] >= TAU).mean()) if (y == 0).any() else np.nan
    return row


# Huong score dang ky TRUOC theo dinh nghia paper (statistical_branch_plan.md §4):
# get_roc_metrics(real_preds, sample_preds) voi real=human nhan 0, samples=AI nhan 1 -> cao = AI.
# KHONG tu lat dau theo nhan val, chi canh bao.
SCORE_DIRECTION = "high=AI"
_auc = auroc(val_df["label"], val_df["score"])
_std = float(np.std(val_df["score"]))
print(f"std(score val) = {_std:.4f} | AUROC(val) = {_auc:.4f} | huong dang ky: {SCORE_DIRECTION}")
if _std < 1e-6:
    print("[!] CANH BAO: score gan nhu hang so - kiem tra lai Agg(MDE)/backbone truoc khi tin so lieu.")
if not np.isnan(_auc) and _auc < 0.5:
    print("[!] CANH BAO: AUROC(val) < 0.5 voi huong da dang ky truoc. "
          "KHONG tu lat dau - kiem tra implementation hoac backbone mismatch.")

_, TAU = tpr_at_fpr(val_df["label"], val_df["score"], TARGET_FPR)
print(f"tau (FPR={TARGET_FPR:.0%} tren val) = {TAU:.4f}")

rows = [r for r in [report_split("val", val_df), report_split("test_ood_model", ood_df),
                    report_split("test_fp", fp_df)] if r]
metrics_df = pd.DataFrame(rows).set_index("split")
pd.set_option("display.width", 200, "display.max_columns", 20)
print(metrics_df.round(4).to_string())

# ── HEADLINE: ĐÚNG 3 số như supervised_models để đặt cạnh nhau đọc được ngay ───
val_auc = metrics_df.loc["val", "auroc"] if "val" in metrics_df.index else np.nan
ood_auc = metrics_df.loc["test_ood_model", "auroc"] if "test_ood_model" in metrics_df.index else np.nan
print("=" * 60)
print(f"HEADLINE — {RUN_NAME} (zero-shot, dual-T4 data-sharding, seed {SEED})")
print("=" * 60)
if "test_ood_model" in metrics_df.index:
    print(f"1) TPR@FPR=1% OOD-model : {metrics_df.loc['test_ood_model', 'tpr@fpr0.01']:.3f}")
if "test_fp" in metrics_df.index:
    print(f"2) FPR thuc @ tau test_fp : {metrics_df.loc['test_fp', 'fpr@tau']:.4f}  "
          f"(mong ~{TARGET_FPR})")
print(f"3) AUROC gap val->OOD   : {val_auc:.3f} -> {ood_auc:.3f}  (delta={val_auc - ood_auc:+.3f})")
print()
print("LUU Y khi dat canh supervised + AL (xem muc 'Cong bang' o markdown dau bai):")
print("  - val: supervised dung val de CHON CHECKPOINT (load_best_model_at_end + early stop),")
print("    Lastde++ chi lay tau tu val => AUROC(val) cua supervised lac quan. KHONG xep hang tren val.")
print("  - Delta(val->OOD) chi co nghia voi supervised: test_ood_model la generator ho")
print("    google/deepseek chua tung thay khi TRAIN; Lastde++ training-free nen khong co in-domain.")
print("  - Lastde++ o day la BLACK-BOX nhung DUNG protocol tac gia (detection_black_box.sh,")
print("    proxy_model=gptj_6b) -> nhe hon caveat cua DetectGPT (gpt2-xl la lua chon ung bien).")
print("  - Quy mo: roberta-large 355M DUOC TRAIN vs gpt-j-6b 6B FROZEN, 1 model, 1 forward pass.")

metrics_out = {"run": RUN_NAME, "model": val_df["_run_signature"].iloc[0].split("|")[1],
               "tau": TAU, "target_fpr": TARGET_FPR,
               "world_size": WORLD_SIZE, "seed": SEED,
               "training_free": True, "flipped_sign": False,
               "impl_version": "lastde-plusplus-paper-faithful-v1",
               "run_signature": val_df["_run_signature"].iloc[0],
               "embed_size": 4, "epsilon_mult": 8, "tau_prime": 15,
               "n_samples": 100, "agg": "std", "max_len": 512,
               "score_direction": SCORE_DIRECTION,
               "metrics": metrics_df.reset_index().to_dict("records")}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2, default=float))
for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if d is not None:
        d[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(
            OUT_DIR / f"pred_{nm}.parquet", index=False)
print("da luu artifacts ->", OUT_DIR)


In [ ]:
# 📊 Tóm tắt nhanh (đọc lại metrics.json vừa lưu)
_p = OUT_DIR / "metrics.json"
if _p.exists():
    m = json.loads(_p.read_text())
    print(f"RUN = {m['run']} | {m['model']}")
    print(f"s={m['embed_size']} eps={m['epsilon_mult']}*n tau'={m['tau_prime']} "
          f"n_samples={m['n_samples']} agg={m['agg']} | tau = {m['tau']:.4f} "
          f"| flipped_sign = {m['flipped_sign']}")
    for r in m["metrics"]:
        print(f"{r['split']:16} AUROC={r['auroc']:.4f}  TPR@FPR1%={r['tpr@fpr0.01']:.4f}  "
              f"TPR@FPR0.1%={r['tpr@fpr0.001']:.4f}  FPR@tau={r['fpr@tau']:.4f}")
else:
    print("Chưa có metrics.json — chạy cell gộp + metrics trước.")


In [ ]:
# 🎲 Tổng hợp MULTI-SEED: mean ± std — cùng dạng báo cáo với supervised_models
# Chay duoc voi 1 seed (in ra std = 0.0000) hoac nhieu seed.
# Muon them seed: chay lai cell FULL voi  RUN_SEED=43  roi  RUN_SEED=44  o dau lenh, vi du:
#   !RUN_SEED=43 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 lastde_worker.py
#   !RUN_SEED=44 OMP_NUM_THREADS=2 torchrun --standalone --nproc_per_node=2 lastde_worker.py
# DUNG 43/44, KHONG dung 1/2: supervised_models bao mean±std tren seed 42/43/44, nen bang
# cuoi cung phai noi duoc 'ca ba nhanh deu 3 seed 42/43/44'. Lech seed thi khong noi duoc.
# Lastde++ full ~1-2.5h/seed nen 3 seed (~3-7.5h) VUA quota 30h/tuan — khac DetectGPT (18h/seed).
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd

RUN_NAME = "lastde_doubleplus_gptj"
_base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
paths = sorted(glob.glob(str(_base / f"artifacts_{RUN_NAME}_seed*" / "metrics.json")))
print(f"Tim thay {len(paths)} seed cho {RUN_NAME}:", [Path(p).parent.name for p in paths])

if not paths:
    print("Chua co seed nao - chay cell FULL + metrics truoc.")
else:
    recs = []
    for p in paths:
        m = json.loads(Path(p).read_text())
        for r in m["metrics"]:
            recs.append({"seed": m["seed"], "tau": m["tau"], **r})
    allm = pd.DataFrame(recs)

    print(f"\n=== Chi tiet tung seed ({RUN_NAME}) ===")
    _show = ["seed", "split", "n", "auroc", "tpr@fpr0.01", "tpr@fpr0.001", "fpr@tau", "tau"]
    print(allm[[c for c in _show if c in allm.columns]].round(4).to_string(index=False))

    METRIC_COLS = [c for c in ["auroc", "auprc", "tpr@fpr0.001", "tpr@fpr0.01",
                               "tpr@fpr0.05", "tpr@tau", "fpr@tau"] if c in allm.columns]
    agg = allm.groupby("split")[METRIC_COLS].agg(["mean", "std"])
    # std cua 1 seed = NaN -> 0.0 (khong co bien thien de do). TUYET DOI KHONG fillna cot mean:
    # test_fp chi co 1 lop nen auroc/auprc that su la NaN — fill 0 se bi doc thanh "AUROC = 0".
    for _c in METRIC_COLS:
        agg[(_c, "std")] = agg[(_c, "std")].fillna(0.0)

    def _fmt(split, c):
        mu, sd = agg.loc[split, (c, "mean")], agg.loc[split, (c, "std")]
        return f"{c}=n/a" if np.isnan(mu) else f"{c}={mu:.4f}+/-{sd:.4f}"

    print(f"\n=== mean +/- std qua {allm['seed'].nunique()} seed  (n/a = khong dinh nghia) ===")
    for split in ["val", "test_ood_model", "test_fp"]:
        if split not in agg.index:
            continue
        print(f"{split:16} " + "  ".join(_fmt(split, c) for c in METRIC_COLS))
    print("\n(test_fp chi co lop human -> auroc/auprc/tpr@fpr la n/a; doc cot fpr@tau cho split nay.)")

    def _num(v):
        return None if np.isnan(v) else float(v)

    out = _base / f"summary_{RUN_NAME}_multiseed.json"
    out.write_text(json.dumps(
        {"run": RUN_NAME, "n_seeds": int(allm["seed"].nunique()),
         "seeds": sorted(allm["seed"].unique().tolist()),
         "tau_mean": float(allm["tau"].mean()),
         "tau_std": float(allm["tau"].std(ddof=1)) if allm["seed"].nunique() > 1 else 0.0,
         "agg": {s: {c: {"mean": _num(agg.loc[s, (c, "mean")]),
                         "std": _num(agg.loc[s, (c, "std")])} for c in METRIC_COLS}
                 for s in agg.index}}, indent=2))
    print("\nda luu ->", out)
    if allm["seed"].nunique() == 1:
        print("[!] Chi co 1 seed -> std = 0 la GIA. Ghi ro 'single-seed' trong bao cao, hoac")
        print("    chay them RUN_SEED=43 roi RUN_SEED=44 (~53ph/seed tren deconf2, vua quota).")

In [ ]:
# ── CHẨN ĐOÁN "CÓ SO SÁNH ĐƯỢC KHÔNG" — chạy sau cell metric ─────────────────
# Bảng metric ở trên không trả lời được ba câu mà bảng benchmark cuối cùng phải trả lời:
#
#   1. CHIỀU có nhất quán giữa các split? val và test_ood_model cùng phía 0.5 -> không có
#      bug đảo dấu. Nằm hai phía -> không có tín hiệu, và đừng đọc dấu của từng split.
#   2. AUROC có khác ĐOÁN MÒ không? SD của AUROC khi detector vô hiệu là
#      sqrt((n1+n2+1)/(12*n1*n2)) — với val 438+438 thì SD = 0.0195, nên 0.48 và 0.52
#      là CÙNG một kết luận: không phân biệt được.
#   3. TPR@FPR=1% so với mốc nào? Mốc đoán mò là ĐÚNG 0.01 (ROC của classifier vô hiệu là
#      đường chéo), KHÔNG phải 0. Nên 0.0337 là 3.4x đoán mò, không phải "gần như trượt".
#
# In cả mean score theo nhãn: nếu mean(AI) > mean(human) ở CẢ BA split thì hướng đăng ký
# trước là đúng và số thấp là bản chất, không phải lỗi implementation.
import glob
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

_CHANCE = {"tpr@fpr0.001": 0.001, "tpr@fpr0.01": 0.01, "tpr@fpr0.05": 0.05}


def _sd_null(n1, n2):
    return math.sqrt((n1 + n2 + 1) / (12 * n1 * n2))


def _p2(z):
    return math.erfc(abs(z) / math.sqrt(2))


# Loại file KHÔNG có nhãn nhị phân:
#   pred_mix*     -> tập trộn, chỉ có mix_ratio_actual liên tục. Đọc nó ở đây là KeyError:'label'.
#   pred_matched* -> tập con đối-shortcut, báo riêng ở notebook supervised.
_EXCL = ("pred_mix", "pred_matched")
_all_preds = sorted(glob.glob("**/pred_*.parquet", recursive=True))
_preds = [p for p in _all_preds if not any(x in Path(p).name for x in _EXCL)]
_skipped = [p for p in _all_preds if p not in _preds]
if _skipped:
    print(f"bỏ qua {len(_skipped)} file không có nhãn nhị phân: "
          f"{[Path(p).name for p in _skipped]}")
    print("  (mix dùng cho đường dose-response ở bước tổng hợp, không tính AUROC)\n")
if not _preds:
    print("khong tim thay pred_*.parquet -> chay cell metric truoc")
else:
    print("=" * 92)
    print("CHIEU SCORE THEO NHAN (mean; huong dang ky: cao = AI)")
    print("=" * 92)
    print(f"  {'split':18}{'n':>7}{'mean human':>13}{'mean AI':>12}{'gap':>11}  dung chieu?")
    _signs = {}
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        if "label" not in _d.columns:      # phòng thêm: file lạ không có nhãn
            print(f"  (bỏ qua {Path(_p).name}: không có cột label)")
            continue
        _sp = Path(_p).stem.replace("pred_", "")
        _h = _d.loc[_d["label"] == 0, "score"]
        _a = _d.loc[_d["label"] == 1, "score"]
        if len(_a) == 0:
            print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{'—':>12}{'—':>11}  (1 lop, bo qua)")
            continue
        _gap = _a.mean() - _h.mean()
        _signs[_sp] = _gap
        print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{_a.mean():12.4f}{_gap:+11.4f}"
              f"  {'CO' if _gap > 0 else 'KHONG'}")
    if len(_signs) >= 2:
        _ok = all(v > 0 for v in _signs.values()) or all(v < 0 for v in _signs.values())
        print(f"\n  -> chieu {'NHAT QUAN' if _ok else 'KHONG nhat quan'} qua "
              f"{len(_signs)} split"
              + ("" if _ok else "  <-- dau hieu KHONG CO TIN HIEU, dung doc dau tung split"))
        if _ok and all(v > 0 for v in _signs.values()):
            print("     dung huong dang ky truoc => implementation dung chieu; so thap la ban chat")

    print()
    print("=" * 92)
    print("CO HON DOAN MO KHONG")
    print("=" * 92)
    print(f"  {'split':18}{'AUROC':>9}{'SD_null':>10}{'z':>8}{'p':>9}  ket luan")
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        if "label" not in _d.columns:      # phòng thêm: file lạ không có nhãn
            print(f"  (bỏ qua {Path(_p).name}: không có cột label)")
            continue
        _sp = Path(_p).stem.replace("pred_", "")
        _n1 = int((_d["label"] == 1).sum())
        _n0 = int((_d["label"] == 0).sum())
        if _n1 == 0 or _n0 == 0:
            continue
        _a = float(roc_auc_score(_d["label"], _d["score"]))
        _sd = _sd_null(_n0, _n1)
        _z = (_a - 0.5) / _sd
        _v = "hon ngau nhien" if (_p2(_z) < 0.05 and _z > 0) else (
            "DUOI ngau nhien" if _p2(_z) < 0.05 else "KHONG phan biet duoc voi doan mo")
        print(f"  {_sp:18}{_a:9.4f}{_sd:10.4f}{_z:+8.2f}{_p2(_z):9.4f}  {_v}")

    print()
    print("=" * 92)
    print("TPR SO MOC DOAN MO (moc = dung bang FPR, khong phai 0)")
    print("=" * 92)
    _art = Path(_preds[0]).parent
    _mp = _art / "metrics.json"
    _m = json.loads(_mp.read_text()) if _mp.exists() else None
    if _m is None:
        print("  khong doc duoc metrics.json")
    else:
        print(f"  {'split':18}{'chi so':16}{'gia tri':>10}{'doan mo':>10}{'lan':>7}{'z':>8}")
        for _r in _m["metrics"]:
            _npos = _r.get("n_ai") or 0
            _nneg = (_r.get("n") or 0) - _npos
            if not _npos or not _nneg:
                continue
            for _k, _c in _CHANCE.items():
                _v = _r.get(_k)
                if _v is None or (isinstance(_v, float) and math.isnan(_v)):
                    continue
                _z = (_v - _c) / math.sqrt(_c * (1 - _c) / _npos)
                # nguong FPR duoc dat boi phan vi tren NEGATIVE -> so mau quyet dinh = _c*_nneg
                _ndec = _c * _nneg
                _flag = f"  <-- nguong do ~{_ndec:.1f} mau quyet dinh, DUNG BAO" if _ndec < 3 else ""
                print(f"  {_r['split']:18}{_k:16}{_v:10.4f}{_c:10.4f}{_v / _c:6.1f}x{_z:+8.1f}{_flag}")